In [1]:
using Revise
using QuantumClifford

┌ Info: Precompiling QuantumClifford [0525e862-1e90-11e9-3e4d-1b39d7109de1]
└ @ Base loading.jl:1423


## About jbig and jsmall

In [10]:
s = ghz(6)
xzs = ghz(6).xzs
Tme = eltype(xzs)
lowbit = Tme(0x1)
zerobit = Tme(0x0)
rows, columns = size(stabilizerview(s))

for j in 1:columns
    jbig = QuantumClifford._div(Tme,j-1)+1
    jsmall = lowbit<<QuantumClifford._mod(Tme,j-1)
    print("$jbig $jsmall\n")
end

1 1
1 2
1 4
1 8
1 16
1 32


## Clipping algorithm

### Goal: clipped gauge

1. $\rho_{\mathtt l}(x) + \rho_{\mathtt r}(x) = 2, \forall x$.
2. For all $x$ s.t. $\rho_{\mathtt l}(x) = 2$ or $\rho_{\mathtt r}(x) = 2$, the two stabilizers should be different.

### Steps

Step 1: pregauge.

1. $\rho_{\mathtt l}(x) \le 2, \forall x$.
2. For all $x$ s.t. $\rho_{\mathtt l}(x) = 2$, the two stabilizers should be different.

Setp 2: guage.

Do same thing to the right ends.

#### Algorithm

A variant of Gaussian elimination.

For the 1st qubit, try to find $g_i$ and $g_j$ nontrivially acting on it and $g_i \neq g_j$.

* If we can find $g_i$ and $g_j$, then use them to eliminate all other stabilizers on the 1st qubit. Note that we alway eliminate the longer stabliziers by shorter ones to make the other end not extended.
* If we cannot find such two stabilizers, then two possible cases:
    * Only one stabilizer acts nontrivially on the 1st qubit, then just pass.
    * More than one stabilizers act nontrivially on the 1st qubit, but all of them share a same Pauli operator on it.  So we can use eliminate all expect one.
    
And repeat the process to other qubits in the way of Gaussian elimination.

### Examples

In [19]:
stest = random_stabilizer(10)

+ _YX_Z_XYX_
+ XXZZYZZXZZ
- Z_XZ__XZZ_
+ _YXXYYX_Y_
- XY__X__ZXZ
+ YZZ___ZZXZ
- ZY_ZZYZZX_
+ _ZXXX_Z_Z_
+ YZ_XXZ_ZZX
- Z____Z_ZZZ

In [20]:
clipped_stest = copy(stest)
halfclip!(clipped_stest)

+ XXZZYZZXZZ
- Z_XZ__XZZ_
+ _YX_Z_XYX_
- _ZZZZZZYY_
- __YYYZ_YX_
- __ZX__ZZYY
+ ___XXY_YZ_
- ____XZ_Z__
+ ____Y_YXZX
- _____YZYZ_

In [21]:
logdot(stest, clipped_stest)

0